In [1]:
import numpy as np 
import tensorflow as tf 
import pandas as pd 
from sklearn.preprocessing import MinMaxScaler 

In [3]:
print("Step 1: Creating Stock Price Dataset") 
data = { 
"Open Price": [100, 102, 101, 103, 105, 107, 108, 110, 109, 111, 112, 114, 
113, 115, 117, 118, 119, 121, 120, 122, 123, 125, 124, 126, 127, 129, 130, 
132, 131, 133], 
"Close Price": [101, 103, 102, 104, 106, 108, 109, 111, 110, 112, 113, 
115, 114, 116, 118, 119, 120, 122, 121, 123, 124, 126, 125, 127, 128, 130, 
131, 133, 132, 134] 
}
df = pd.DataFrame(data) 
print(df.head())

Step 1: Creating Stock Price Dataset
   Open Price  Close Price
0         100          101
1         102          103
2         101          102
3         103          104
4         105          106


In [5]:
print("Normalizing Data")
scaler=MinMaxScaler()
scaled_data=scaler.fit_transform(df)
print(scaled_data[:5])

Normalizing Data
[[0.         0.        ]
 [0.06060606 0.06060606]
 [0.03030303 0.03030303]
 [0.09090909 0.09090909]
 [0.15151515 0.15151515]]


In [7]:
print("Step 3: Preparing Sequences")
seq_length=3
X_seq=[scaled_data[i:i+seq_length] for i in range(len(scaled_data)-seq_length)]
y_seq = [scaled_data[i+seq_length] for i in range(len(scaled_data) - 
seq_length)] 
X_seq, y_seq = np.array(X_seq), np.array(y_seq) 
print("Sample X_seq:", X_seq[:2]) 
print("Sample y_seq:", y_seq[:2])

Step 3: Preparing Sequences
Sample X_seq: [[[0.         0.        ]
  [0.06060606 0.06060606]
  [0.03030303 0.03030303]]

 [[0.06060606 0.06060606]
  [0.03030303 0.03030303]
  [0.09090909 0.09090909]]]
Sample y_seq: [[0.09090909 0.09090909]
 [0.15151515 0.15151515]]


In [9]:
print("Step 4: Building and Training GRU Model") 
model = tf.keras.Sequential([ 
tf.keras.layers.GRU(50, input_shape=(seq_length, 2)), 
tf.keras.layers.Dense(2) 
]) 
model.compile(optimizer='adam', loss='mse') 
model.fit(X_seq, y_seq, epochs=20, batch_size=8, verbose=0) 
print("Model training complete.")

Step 4: Building and Training GRU Model


C:\Users\K Jyothsna\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model training complete.


In [11]:
print("Step 5: Predicting Next Day Stock Prices") 
predicted_price = scaler.inverse_transform(model.predict(X_seq[-1].reshape(1, 
seq_length, 2))) 
actual_price = scaler.inverse_transform(y_seq[-1].reshape(1, -1)) 
error = np.abs(predicted_price - actual_price) 
print("Predicted (Open, Close):", predicted_price[0]) 
print("Actual (Open, Close):", actual_price[0]) 
print("Error (Open, Close):", error[0])

Step 5: Predicting Next Day Stock Prices
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 458ms/step
Predicted (Open, Close): [132.00099 132.70947]
Actual (Open, Close): [133. 134.]
Error (Open, Close): [0.99900818 1.29052734]
